In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

import sys
import argparse
from PIL import Image
from tqdm import tqdm
from pathlib import Path
import random
import time
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from torchvision import transforms
import torch.optim as optim
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import defaultdict, Counter
import warnings
import models
from torch.utils.data.sampler import WeightedRandomSampler
from matplotlib import rcParams
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from datasets import DamageDataset, ImagePadder, FixedHeightResizeAndPad, make_jitter_transform, TypeAugmenter

In [ ]:
# bash create_labeled_csv.sh
data_csv = Path('damage_classifier_data/dldt_ground_truth/labeled_dataset.csv')
# read csv in and name columns as path and label
df = pd.read_csv(data_csv, names=['path', 'label'])
df

In [ ]:
# extract bookstr and character from path
df['bookstr'] = df['path'].apply(lambda x: Path(x).name.split('-')[0])
df['character'] = df['path'].apply(lambda x: Path(x).name.split("_")[7])
df

In [ ]:
# for every path with label==1, match it with one of its counterparts (e.g., with same bookstr, character, and label==0)
matched_pairs = []

df_neg = df[df['label'] == 0]

no_counterpart_list = []
for idx, row in df[df['label'] == 1].iterrows():
    counterparts = df_neg[(df_neg['bookstr'] == row['bookstr']) & (df_neg['character'] == row['character']) & (df_neg['label'] == 0) & (df_neg['path'] != row['path'])]
    # print(len(counterparts))
    if len(counterparts) == 0:
        no_counterpart_list.append(row['path'])
        continue
    else:
        counterpart = counterparts.sample(n=1).iloc[0]
        # print(counterpart)
        # break
        # remove counterpart from df
        assert counterpart['path'] != row['path']
        assert row['label'] == 1
        assert counterpart['label'] == 0
        df_neg = df_neg[df_neg['path'] != counterpart['path']]
        matched_pairs.append((row['path'], counterpart['path']))

print(f"Number of paths with no counterpart: {len(no_counterpart_list)}")
matched_pairs[:5]

In [ ]:
pos = [x[0] for x in matched_pairs]
neg = [x[1] for x in matched_pairs]
assert len(set(pos).intersection(set(neg))) == 0
len(pos), len(neg), len(set(pos)), len(set(neg)), len(set(pos).intersection(set(neg))), len(set(pos).union(set(neg)))

In [ ]:
# split into 90/10 train/test
random.seed(666)
random.shuffle(matched_pairs)
train_pairs = matched_pairs[:int(len(matched_pairs)*0.9)]
test_pairs = matched_pairs[int(len(matched_pairs)*0.9):]
len(train_pairs), len(test_pairs)

In [ ]:
train_pairs

In [ ]:
# dump train and test pairs to separate csv files
train_csv = Path('damage_classifier_data/dldt_ground_truth/train_pairs.csv')
test_csv = Path('damage_classifier_data/dldt_ground_truth/test_pairs.csv')
train_df_pairs = pd.DataFrame(train_pairs, columns=['pos', 'neg'])
train_df_pairs.to_csv(train_csv, index=False)
test_df_pairs = pd.DataFrame(test_pairs, columns=['pos', 'neg'])
test_df_pairs.to_csv(test_csv, index=False)

# also dump train and test csvs for all images
train_csv = Path('damage_classifier_data/dldt_ground_truth/train.csv')
test_csv = Path('damage_classifier_data/dldt_ground_truth/test.csv')
train_pos = [{'path': x[0], 'label': 1} for x in train_pairs]
train_neg = [{'path': x[1], 'label': 0} for x in train_pairs]
train_df = pd.DataFrame(train_pos + train_neg)
train_df.to_csv(train_csv, index=False)
test_pos = [{'path': x[0], 'label': 1} for x in test_pairs]
test_neg = [{'path': x[1], 'label': 0} for x in test_pairs]
test_df = pd.DataFrame(test_pos + test_neg)
test_df.to_csv(test_csv, index=False)


In [ ]:
!wc -l damage_classifier_data/dldt_ground_truth/*.csv
!head -n 1 damage_classifier_data/dldt_ground_truth/*.csv

In [ ]:
# show each pair image side by side
def show_pair(pair):
    img1 = Image.open(pair[0])
    img2 = Image.open(pair[1])
    fig, ax = plt.subplots(1, 2, figsize=(10, 5))
    ax[0].imshow(img1)
    ax[0].set_title('Positive')
    ax[1].imshow(img2)
    ax[1].set_title('Negative')
    plt.show()

show_pair(train_pairs[0])
show_pair(train_pairs[1])
show_pair(train_pairs[2])
show_pair(train_pairs[3])
show_pair(train_pairs[4])
show_pair(train_pairs[5])
show_pair(train_pairs[6])


In [ ]:
from matcher import get_transform
import torchvision

In [ ]:
failing_img = '/graft2/code/nvog/git/matching/damage_classifier_data/dldt_ground_truth/imgs/newcombjr_R19638_otstm_2_completehistoryofengland1685-160_page1rline50_char30_G_uc.tif'
img = Image.open(failing_img)
img

In [ ]:
transform = get_transform()
img_tensor = transform(img)
torchvision.transforms.ToPILImage()(img_tensor)

In [ ]:
img_tensor

In [ ]:
transform

In [ ]:
from make_twin_dataset_from_splits import SquarePad, Binarizer
import skimage

In [ ]:
SquarePad()(img)

In [ ]:
torchvision.transforms.Resize((112, 112))(SquarePad()(img))

In [ ]:
gs_img = transforms.Grayscale(num_output_channels=1)(torchvision.transforms.Resize((112, 112))(SquarePad()(img)))
gs_img

In [ ]:
Binarizer()(gs_img)

In [ ]:
# print(img.mode)
Image.fromarray((np.array(gs_img) > skimage.filters.threshold_local(np.array(gs_img), 15, offset=10)))

In [ ]:
Image.fromarray((np.array(gs_img) > skimage.filters.threshold_otsu(np.array(gs_img))))

In [ ]:
gs_img_arr = np.array(gs_img)
# mask out white pixels from gs_img
# gs_img_arr_no_white = np.where(gs_img_arr < 255, gs_img_arr, 100)
# print(gs_img_arr_no_white.shape)
no_white_mask = gs_img_arr < 255
thresh_min = skimage.filters.threshold_otsu(gs_img_arr[no_white_mask].reshape(-1, 1))
binary_min = gs_img_arr > thresh_min
binary_min[~no_white_mask] = 255

fig, ax = plt.subplots(2, 2, figsize=(10, 10))

ax[0, 0].imshow(gs_img_arr, cmap=plt.cm.gray)
ax[0, 0].set_title('Original')

ax[0, 1].hist(gs_img_arr.ravel(), bins=256)
ax[0, 1].set_title('Histogram')

ax[1, 0].imshow(binary_min, cmap=plt.cm.gray)
ax[1, 0].set_title('Thresholded (min)')

ax[1, 1].hist(gs_img_arr.ravel(), bins=256)
ax[1, 1].axvline(thresh_min, color='r')

for a in ax[:, 0]:
    a.set_axis_off()

plt.show()